A set of simulations & hypothesis tests for assessing the statistical power of "minP vs CRE of interest" tests in the shendure dataset...

Essentially a better version of `power_shendure_vs_minp.ipynb` with a more representative distribution of positive and negative effects. Specifically, we will be using the real values, plus many negatives.

Based on UKBB paper, expect ~30% of library to be active. So we will add 2x original size of negatives... 

We will also reduce computational burden by producing half-orthos.

# Imports & dask cluster creation

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
from pathlib import Path

%load_ext autoreload
%autoreload 2

2026-01-26 12:30:15.330986: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-26 12:30:15.334178: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="32G",#memory per slurm job
        processes=1,#dask workers per slurm jobTrueT
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=4:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=2)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

In [3]:
client.dashboard_link

'http://127.0.0.1:8787/status'

# Ground truth creation

We will use parameter estimates averaged across cell-type nd cre models. 

When we perform the simulation, we are going to IGNORE cell type!

The shendure dataset is characterized by extremely heterogenious transfection, so different sets of CREs are represented in different cell types, likely due to differential clonotype contribution to different cell-types. For this reason, we don't have ground truth values for many combinations of cre_id, cell type. This means that direct simulation runs into problems, since it allows transfection of any cre into any cell type...

We don't care about cell types in this analysis, so we are just going to remove that information and treat the same cre transformed into two different cell-types as two different entities...

See commit `58a65def6964cea9247c15937dd60714489f1750` and 2026-10-23 notes for further discussion.

In [4]:
data_root=Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")

In [5]:
primordial=scm.ortho.load(client,data_root/"shendure","ortho_primordial_v4")
primordial.compute_model_qc()

We use by_cell_type models, expecting that they will have more robust estimates...

In [6]:
import pandas as pd
import numpy as np

vals=[]
for key in primordial.by_cell_qc.keys():
    working=primordial.by_cell_qc[key]['dat'].reset_index().drop(columns=["mean(umis_mpra_bc)"])
    working["cell_type"]=key
    vals.append(working)
gt_cell_type=pd.concat(vals)

In [7]:
#casting away from sparse, since it's not sparse anymore
gt_cell_type["mu"] = gt_cell_type["mu"].astype(float)

#this doesn't even work
gt_cell_type["cre_id"]=gt_cell_type["cre_id"].astype(str)
gt_cell_type["cell_type"]=gt_cell_type["cell_type"].astype(str)

In [8]:
gt_cell_type.dtypes

cre_id        object
mu           float64
cell_type     object
dtype: object

In [9]:
gt_cell_type

,cre_id,mu,cell_type
0,Bend5_chr4_8168,0.019097,SurfaceEctoderm
1,Bend5_chr4_8174,0.011630,SurfaceEctoderm
2,Bend5_chr4_8175,0.671256,SurfaceEctoderm
3,Bend5_chr4_8179,0.019683,SurfaceEctoderm
4,Bend5_chr4_8199,0.004408,SurfaceEctoderm
...,...,...,...
80,Txndc12_chr4_7978,0.995304,NeuroectodermRostral
81,eef1aP,44.205127,NeuroectodermRostral
82,pgk1P,9.947188,NeuroectodermRostral
83,reference,0.015677,NeuroectodermRostral


Now, we discard cell-type information, as discussed above.

In [10]:
gt_cell_type["cre_id"] = gt_cell_type["cre_id"] + "---" + gt_cell_type["cell_type"]
gt_cell_type["cell_type"] = "reference"
gt_cell_type

,cre_id,mu,cell_type
0,Bend5_chr4_8168---SurfaceEctoderm,0.019097,reference
1,Bend5_chr4_8174---SurfaceEctoderm,0.011630,reference
2,Bend5_chr4_8175---SurfaceEctoderm,0.671256,reference
3,Bend5_chr4_8179---SurfaceEctoderm,0.019683,reference
4,Bend5_chr4_8199---SurfaceEctoderm,0.004408,reference
...,...,...,...
80,Txndc12_chr4_7978---NeuroectodermRostral,0.995304,reference
81,eef1aP---NeuroectodermRostral,44.205127,reference
82,pgk1P---NeuroectodermRostral,9.947188,reference
83,reference---NeuroectodermRostral,0.015677,reference


In [11]:
#sanity check
assert len(gt_cell_type) == len(gt_cell_type["cre_id"].unique())
len(gt_cell_type)

1462

Now that we have reasonable mu estimates for the real CRE, let us add 200% "indistinguishable from minP".

In [12]:
inactive_names=["inactive_"+str(i) for i in range(0,len(gt_cell_type)*2)]
inactive_df=pd.DataFrame({"cre_id":inactive_names})
minP=scm.SHENDURE_BOUNDS.reference_activity
inactive_df["mu"]=minP
inactive_df["cell_type"]="reference"
inactive_df


,cre_id,mu,cell_type
0,inactive_0,0.019311,reference
1,inactive_1,0.019311,reference
2,inactive_2,0.019311,reference
3,inactive_3,0.019311,reference
4,inactive_4,0.019311,reference
...,...,...,...
2919,inactive_2919,0.019311,reference
2920,inactive_2920,0.019311,reference
2921,inactive_2921,0.019311,reference
2922,inactive_2922,0.019311,reference


Then stack with original gt...

In [13]:
final_gt=pd.concat([gt_cell_type,inactive_df],ignore_index=True).rename({"mu":"true_mean"},axis=1)
final_gt

,cre_id,true_mean,cell_type
0,Bend5_chr4_8168---SurfaceEctoderm,0.019097,reference
1,Bend5_chr4_8174---SurfaceEctoderm,0.011630,reference
2,Bend5_chr4_8175---SurfaceEctoderm,0.671256,reference
3,Bend5_chr4_8179---SurfaceEctoderm,0.019683,reference
4,Bend5_chr4_8199---SurfaceEctoderm,0.004408,reference
...,...,...,...
4381,inactive_2919,0.019311,reference
4382,inactive_2920,0.019311,reference
4383,inactive_2921,0.019311,reference
4384,inactive_2922,0.019311,reference


In [14]:
#sanity check
assert len(final_gt[["cre_id","cell_type"]].drop_duplicates()) == len(final_gt)

# Creating artificial libraries

In [15]:
libraries=[scm.simulate_library(CREs=final_gt["cre_id"],
                 library_model=scm.SHENDURE_BOUNDS.library_model)
                 for i in range(5)]

In [16]:
libraries[2]

,cre_id,mpra_bc,abundance
0,Bend5_chr4_8168---SurfaceEctoderm,AAAAAAAAAAAAAAAAAAAA,8.124710e-07
1,Bend5_chr4_8168---SurfaceEctoderm,AAAAAAAAAAAAAAAAAAAC,2.345259e-06
2,Bend5_chr4_8168---SurfaceEctoderm,AAAAAAAAAAAAAAAAAAAG,9.066642e-07
3,Bend5_chr4_8168---SurfaceEctoderm,AAAAAAAAAAAAAAAAAAAT,2.046199e-06
4,Bend5_chr4_8168---SurfaceEctoderm,AAAAAAAAAAAAAAAAAACA,4.580546e-07
...,...,...,...
595315,inactive_2923,AAAAAAAAAAGCACCCCTAT,9.956462e-07
595316,inactive_2923,AAAAAAAAAAGCACCCCTCA,7.687381e-07
595317,inactive_2923,AAAAAAAAAAGCACCCCTCC,5.529782e-06
595318,inactive_2923,AAAAAAAAAAGCACCCCTCG,4.645112e-06


# Creating sim

Create some bounds to simulate from. These will be identical to the normal shendure bounds, except that we simplify to just one cell-type...

In [17]:
bound=scm.SHENDURE_BOUNDS.copy()

In [18]:
print(bound.cells_per_cell_type.name)
print(bound.cells_per_cell_type.index.name)
bound.cells_per_cell_type

cells_per_cell_type
cell_type


cell_type
Cardiomyocytes              680
EpiblastPrimitiveStreak    3445
ExEndodermParietal         4644
ExEndodermVisceral         3238
Haematoendothelial         1079
Mesoderm                   7427
NeuroectodermBrain         7750
NeuroectodermRostral       1757
SurfaceEctoderm            5168
reference                  8201
Name: cells_per_cell_type, dtype: int64

In [19]:
type(bound.cells_per_cell_type)

pandas.core.series.Series

In [20]:
working=pd.Series({"reference":bound.cells_per_cell_type.sum()})
working.name=bound.cells_per_cell_type.name
working.index.name=bound.cells_per_cell_type.index.name
working

cell_type
reference    43389
Name: cells_per_cell_type, dtype: int64

In [21]:
bound.cells_per_cell_type=working

In [22]:
type(bound.cells_per_cell_type)

pandas.core.series.Series

In [23]:
sim=scm.de_novo_simulation(location=data_root,
                            name="twothird_pow_sim_2026-01-26",
                            client=client,
                            libraries=libraries,
                            library_mapping="corresponding",
                            n_sims=5,
                            experiment_bounds=bound,
                            ground_truth=final_gt)

scMPRAforge: INFO: 'state.parquet' found for 'twothird_pow_sim_2026-01-26', loading.


In [ ]:
sim.gamut()

scMPRAforge: INFO: 43389
scMPRAforge: INFO: <class 'numpy.int64'>
2026-01-26 12:30:40,908 - distributed.worker - ERROR - Compute Failed
Key:       _simulate_transfection_helper-d0e57e7aa177a2aba73f476205ad73e9
State:     executing
Task:  <Task '_simulate_transfection_helper-d0e57e7aa177a2aba73f476205ad73e9' _simulate_transfection_helper(, ...)>
Exception: 'AttributeError("\'numpy.int64\' object has no attribute \'name\'")'
Traceback: '  File "/nfs/roberts/project/pi_skr2/mcn26/tabula-rasa/notebooks/object_creation/simulations/emperically_calibrated/shendure/scMPRAforge/core.py", line 5095, in _simulate_transfection_helper\n    transfected=_simulate_transfection(\n  File "/nfs/roberts/project/pi_skr2/mcn26/tabula-rasa/notebooks/object_creation/simulations/emperically_calibrated/shendure/scMPRAforge/core.py", line 5850, in _simulate_transfection\n    working=_simulate_single_replicate_transfection()\n  File "/nfs/roberts/project/pi_skr2/mcn26/tabula-rasa/notebooks/object_creation/simulat

2026-01-26 12:30:41,878 - distributed.worker - ERROR - Compute Failed
Key:       _simulate_transfection_helper-e4211f6d9a2372d0038457d72de9eae6
State:     executing
Task:  <Task '_simulate_transfection_helper-e4211f6d9a2372d0038457d72de9eae6' _simulate_transfection_helper(, ...)>
Exception: 'AttributeError("\'numpy.int64\' object has no attribute \'name\'")'
Traceback: '  File "/nfs/roberts/project/pi_skr2/mcn26/tabula-rasa/notebooks/object_creation/simulations/emperically_calibrated/shendure/scMPRAforge/core.py", line 5095, in _simulate_transfection_helper\n    transfected=_simulate_transfection(\n  File "/nfs/roberts/project/pi_skr2/mcn26/tabula-rasa/notebooks/object_creation/simulations/emperically_calibrated/shendure/scMPRAforge/core.py", line 5850, in _simulate_transfection\n    working=_simulate_single_replicate_transfection()\n  File "/nfs/roberts/project/pi_skr2/mcn26/tabula-rasa/notebooks/object_creation/simulations/emperically_calibrated/shendure/scMPRAforge/core.py", line 57

In [25]:
#sim.save()

In [26]:
#sim

2026-01-26 12:30:42,065 - distributed.worker - ERROR - Compute Failed
Key:       _simulate_transfection_helper-67866f2032b2b81a04f22d0112d1a20e
State:     executing
Task:  <Task '_simulate_transfection_helper-67866f2032b2b81a04f22d0112d1a20e' _simulate_transfection_helper(, ...)>
Exception: 'AttributeError("\'numpy.int64\' object has no attribute \'name\'")'
Traceback: '  File "/nfs/roberts/project/pi_skr2/mcn26/tabula-rasa/notebooks/object_creation/simulations/emperically_calibrated/shendure/scMPRAforge/core.py", line 5095, in _simulate_transfection_helper\n    transfected=_simulate_transfection(\n  File "/nfs/roberts/project/pi_skr2/mcn26/tabula-rasa/notebooks/object_creation/simulations/emperically_calibrated/shendure/scMPRAforge/core.py", line 5850, in _simulate_transfection\n    working=_simulate_single_replicate_transfection()\n  File "/nfs/roberts/project/pi_skr2/mcn26/tabula-rasa/notebooks/object_creation/simulations/emperically_calibrated/shendure/scMPRAforge/core.py", line 57

Make the hypotheses...

In [27]:
#spread_hypothesis.to_tsv(f"{data_root}/pow_sim_2026-01-03_hypo.tsv")
#hs_all_cre = scm.make_all_by_cre_hypotheses(
#    counts=demo_counts,
#    reference_cell_type="reference",
#)

In [28]:
#client.close()
#cluster.close()

In [29]:
#cells_df=scm.load_df_pickle_debug("permerge_cells_df_2a93a57f821d4c9b95cf54fb6a215508.pkl")
##cells_df=scm.cast_string_keys(cells_df,["cell_type", "cre_id"])
#ground_truth=scm.load_df_pickle_debug("premerge_gt_ad527259c2c743cdaed70e6a02e88a0f.pkl")
##ground_truth=scm.cast_string_keys(ground_truth,["cell_type", "cre_id"])

In [30]:
#cells_df.merge(ground_truth,
#                on=["cell_type","cre_id"],
#                validate="many_to_one",
#                how="left",
#                indicator=True)

2026-01-26 12:30:42,976 - distributed.worker - ERROR - Compute Failed
Key:       _simulate_transfection_helper-bcbe5c935cc143b99aa794402500ee8e
State:     executing
Task:  <Task '_simulate_transfection_helper-bcbe5c935cc143b99aa794402500ee8e' _simulate_transfection_helper(, ...)>
Exception: 'AttributeError("\'numpy.int64\' object has no attribute \'name\'")'
Traceback: '  File "/nfs/roberts/project/pi_skr2/mcn26/tabula-rasa/notebooks/object_creation/simulations/emperically_calibrated/shendure/scMPRAforge/core.py", line 5095, in _simulate_transfection_helper\n    transfected=_simulate_transfection(\n  File "/nfs/roberts/project/pi_skr2/mcn26/tabula-rasa/notebooks/object_creation/simulations/emperically_calibrated/shendure/scMPRAforge/core.py", line 5850, in _simulate_transfection\n    working=_simulate_single_replicate_transfection()\n  File "/nfs/roberts/project/pi_skr2/mcn26/tabula-rasa/notebooks/object_creation/simulations/emperically_calibrated/shendure/scMPRAforge/core.py", line 57